# 00 Quickstart: loads both datasets and scores the majority-class (ZeroR) baseline with the shared evaluation code

In [1]:
import sys; sys.path.append('..')  # makes `src` importable from notebooks/
import numpy as np
from sklearn.dummy import DummyClassifier
from src.data import load_fma, fma_xy, load_spec_index
from src.evaluate import evaluate, save_result, results_table

In [2]:
fma = load_fma()
X_train, y_train = fma_xy(fma, 'training')
X_test, y_test = fma_xy(fma, 'test')
print(X_train.shape, X_test.shape)
y_train.value_counts()

(6400, 518) (800, 518)


genre
Hip-Hop          800
Pop              800
Folk             800
Rock             800
Experimental     800
International    800
Electronic       800
Instrumental     800
Name: count, dtype: int64

In [3]:
# ZeroR always predicts the most frequent training genre: the floor every model must beat.
zeror = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
metrics, per_class = evaluate(y_test, zeror.predict(X_test), zeror.predict_proba(X_test), labels=zeror.classes_)
save_result('zeror', 'fma', metrics, notes='majority-class reference')
metrics

{'accuracy': 0.125,
 'macro_precision': 0.0156,
 'macro_recall': 0.125,
 'macro_f1': 0.0278,
 'macro_fpr': 0.125,
 'mcc': 0.0,
 'macro_auc_ovr': 0.5}

In [4]:
# ZeroR ignores the inputs, so only the spectrogram labels are needed here.
spec = load_spec_index()
train, test = spec[spec.split == 'training'], spec[spec.split == 'test']
zeror = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(train), 1)), train.genre)
metrics, per_class = evaluate(test.genre, zeror.predict(np.zeros((len(test), 1))),
                              zeror.predict_proba(np.zeros((len(test), 1))), labels=zeror.classes_)
save_result('zeror', 'spec', metrics, notes='majority-class reference')
metrics

{'accuracy': 0.1668,
 'macro_precision': 0.0278,
 'macro_recall': 0.1667,
 'macro_f1': 0.0477,
 'macro_fpr': 0.1667,
 'mcc': 0.0,
 'macro_auc_ovr': 0.5}

In [5]:
results_table('fma')

,accuracy,macro_precision,macro_recall,macro_f1,macro_fpr,mcc,macro_auc_ovr
model,,,,,,,
zeror,0.125,0.0156,0.125,0.0278,0.125,0.0,0.5
